In [ ]:

from zive_metadata_utils import _sum_noise_samples, _summarize_rpeaks
from zive_data_utils import list_ecg_records, read_json_file, load_ecg_npy, bandpass_filter, get_rpeaks
from typing import Dict, Any


dir_folder = "DATA_FOR_TRAINING"

scan_result = list_ecg_records(
    folder=dir_folder,
    data_format="auto",
)

records = scan_result.records
summary = scan_result.summary

print(f"total_json     : {summary.total_json}")
print(f"matched        : {summary.matched}")
print(f"unmatched_json : {summary.unmatched_json}")
print(f"records returned: {len(records)}")

# records = list_ecg_records(folder, data_format="auto")

print(f"Found {len(records)} matched records")

print("\nTesting first 5 records:")
fs = 200  # Sampling frequency in Hz
   
for rec in records[:5]:
    print(f"\nRecord: {rec.ecg_path.name}, {rec.json_path.name}")
    
    if rec.ecg_path is None:
        msg = f"No matching ECG file for JSON '{rec.json_path.name}'"
        continue
    
    signal = load_ecg_npy(rec.ecg_path)
    n_samples = int(signal.shape[0])
    print("Record length:", n_samples)
    
    print("Bandpass 0.5 - 40 Hz filtering...")
    signal_flt = bandpass_filter(signal, fs=200, lowcut=0.5, highcut=40., method="butterworth", order=4)
    
    # print("Finding Rpeaks...")
    rpeaks, _ = get_rpeaks(signal_flt, fs)
    rpeaks = rpeaks.astype(int).tolist()
    print(f"Found {len(rpeaks)} Rpeaks: {rpeaks[:10]} ..." )

    print("Reading JSON metadata...")
    metadata: Dict[str, Any] = {}
    metadata = read_json_file(rec.json_path)
    h_noises = metadata.get("noises_annotated") if isinstance(metadata, dict) else None
    h_nz_cnt, h_nz_samples = _sum_noise_samples(h_noises, fs)
    print(f"Noises annotated (intervals, samples): {h_nz_cnt}, {h_nz_samples}")
    rpeaks_any = metadata.get("rpeaks") if isinstance(metadata, dict) else None
    rpeaks_count, *_ = _summarize_rpeaks(rpeaks_any)
    print(f"Rpeaks from metadata: {rpeaks_count}")


total_json     : 35
matched        : 35
unmatched_json : 0
records returned: 35
Found 35 matched records

Testing first 5 records:

Record: 1001_6.npy, 1001_6.json
Record length: 128000
Bandpass 0.5 - 40 Hz filtering...
Found 721 Rpeaks: [170, 351, 530, 711, 890, 1066, 1247, 1433, 1621, 1805] ...
Reading JSON metadata...
Noises annotated (intervals, samples): 1, 300
Rpeaks from metadata: 720

Record: 1001_8.npy, 1001_8.json
Record length: 128000
Bandpass 0.5 - 40 Hz filtering...
Found 739 Rpeaks: [119, 290, 458, 624, 787, 953, 1121, 1288, 1457, 1629] ...
Reading JSON metadata...
Noises annotated (intervals, samples): 1, 600
Rpeaks from metadata: 739

Record: 1004_0.npy, 1004_0.json
Record length: 127999
Bandpass 0.5 - 40 Hz filtering...
Found 612 Rpeaks: [128, 343, 563, 784, 1008, 1231, 1450, 1665, 1878, 2085] ...
Reading JSON metadata...
Noises annotated (intervals, samples): 6, 8400
Rpeaks from metadata: 612

Record: 1005_0.npy, 1005_0.json
Record length: 127999
Bandpass 0.5 - 40 Hz 